# Principal Component Analysis (PCA) - Problem Formulation

## 1. The Core Objective of PCA
We know PCA reduces dimensions by projecting data onto a lower-dimensional space. But mathematically, what is it trying to achieve? 

The core objective of PCA is to **find a unit vector (Principal Component) such that when we project our original data points onto this vector, the variance of the projected points is maximized.** 

If we have a 3D dataset and want to reduce it to 2D, we need to find two such vectors (PC1 and PC2).

## 2. The Mathematical Steps of PCA

To achieve this maximum variance projection, PCA follows a strict set of linear algebra steps:

### Step 1: Mean Centering
Before doing anything, the data must be centered around the origin `(0,0)`. 
* We calculate the mean of every column.
* We subtract the mean from every data point in that column.
* *Result:* The data's shape and variance remain exactly the same, but the new mean of the dataset is exactly zero.

### Step 2: Covariance Matrix
Variance only tells us the spread of a single column. **Covariance** tells us how two columns vary *together* (e.g., if X goes up, does Y go up or down?). 
* We calculate the Covariance Matrix for the dataset.
* For a 3D dataset (3 features), this results in a $3 \times 3$ matrix.
* The diagonal elements of this matrix represent the **Variance** of individual features.
* The non-diagonal elements represent the **Covariance** between feature pairs.
* *Why?* This single matrix now holds the complete mathematical summary of the data's spread and orientation.

### Step 3: Eigen Decomposition (Eigenvalues & Eigenvectors)
This is the heart of PCA. We perform Eigen Decomposition on the Covariance Matrix.
* **Eigenvectors:** These represent directions (vectors) in space. When a linear transformation is applied, these special vectors do not change their direction. In PCA, these become our **Principal Components**.
* **Eigenvalues:** Every Eigenvector has a corresponding Eigenvalue. This scalar number represents the magnitude of the variance along its Eigenvector.

### Step 4: Selecting Principal Components
* If we had 3 original features, we get 3 Eigenvectors and 3 Eigenvalues.
* We sort the Eigenvectors based on their Eigenvalues in descending order.
* The Eigenvector with the **highest Eigenvalue** is PC1 (it captures the most variance). The second highest is PC2, and so on.
* If we want to reduce the data to 2D, we simply pick the top 2 Eigenvectors and discard the rest.

### Step 5: Transforming the Data
Finally, we take our original mean-centered dataset and take the **dot product** of it with the transposed matrix of our selected Eigenvectors.
* *Result:* The dataset is officially mapped onto the new, lower-dimensional Principal Component axes!

In [1]:
import numpy as np
import pandas as pd

np.random.seed(23) 

mu_vec1 = np.array([0,0,0])
cov_mat1 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

df = pd.DataFrame(class1_sample,columns=['feature1','feature2','feature3'])
df['target'] = 1

mu_vec2 = np.array([1,1,1])
cov_mat2 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df1 = pd.DataFrame(class2_sample,columns=['feature1','feature2','feature3'])

df1['target'] = 0

df = pd.concat([df, df1], ignore_index=True)

df = df.sample(40)

In [2]:
df.head()

,feature1,feature2,feature3,target
2,-0.367548,-1.137460,-1.322148,1
34,0.177061,-0.598109,1.226512,0
14,0.420623,0.411620,-0.071324,1
11,1.968435,-0.547788,-0.679418,1
12,-2.506230,0.146960,0.606195,1


In [3]:
import plotly.express as px

#y_train_trf = y_train.astype(str)
fig = px.scatter_3d(df, x=df['feature1'], y=df['feature2'], z=df['feature3'],
              color=df['target'].astype('str'))
fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))

fig.show()

In [4]:
# Step 1 - Apply standard scaling (mean centering)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df.iloc[:,0:3] = scaler.fit_transform(df.iloc[:,0:3])

In [5]:
# Step 2 - Find Covariance Matrix
covariance_matrix = np.cov([df.iloc[:,0],df.iloc[:,1],df.iloc[:,2]])
print('Covariance Matrix:\n', covariance_matrix)

Covariance Matrix:
 [[1.02564103 0.20478114 0.080118  ]
 [0.20478114 1.02564103 0.19838882]
 [0.080118   0.19838882 1.02564103]]


In [6]:
# Step 3 - Finding EV and EVs
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

In [ ]:
eigen_values

array([1.3536065 +0.j, 0.94557084+0.j, 0.77774573+0.j])

In [8]:
eigen_vectors

array([[-0.53875915+0.j, -0.69363291+0.j,  0.47813384+0.j],
       [-0.65608325+0.j, -0.01057596+0.j, -0.75461442+0.j],
       [-0.52848211+0.j,  0.72025103+0.j,  0.44938304+0.j]])

In [10]:
pc = eigen_vectors[0:2]
pc

array([[-0.53875915+0.j, -0.69363291+0.j,  0.47813384+0.j],
       [-0.65608325+0.j, -0.01057596+0.j, -0.75461442+0.j]])

In [11]:
transformed_df = np.dot(df.iloc[:,0:3],pc.T)
# 40,3 - 3,2
new_df = pd.DataFrame(transformed_df,columns=['PC1','PC2'])
new_df['target'] = df['target'].values
new_df.head()

,PC1,PC2,target
0,0.599433+0.000000j,1.795862+0.000000j,1
1,1.056919+0.000000j,-0.212737+0.000000j,0
2,-0.271876+0.000000j,0.498222+0.000000j,1
3,-0.621586+0.000000j,0.023110+0.000000j,1
4,1.567286+0.000000j,1.730967+0.000000j,1
